# 데이터 엔지니어링·BI 실전 1편: 국민연금 사업장 데이터 첫 점검

[블로그 글](https://datapopcorn.ai/blog/nps-workplace-data-first-look)의 실습을 그대로 재현합니다. 글과 같은 파일(`국민연금 가입 사업장 내역_20261023.csv`, 자료생성년월 2026-09)을 공공데이터포털에서 받습니다.

In [ ]:
!pip -q install duckdb

In [ ]:
import hashlib, urllib.request

# 글에서 쓴 2026-10-23 등록 파일 (매달 새 파일이 올라오므로 파일 ID로 고정)
URL = "https://www.data.go.kr/cmm/cmm/fileDownload.do?atchFileId=FILE_000000007740322&fileDetailSn=1&insertDataPrcus=N"
CSV = "nps_20261023.csv"
req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=300) as r, open(CSV, "wb") as f:
    f.write(r.read())
print(hashlib.sha256(open(CSV, "rb").read()).hexdigest())  # 글에서 쓴 파일과 같은지 확인

## Step 2. 원본은 그대로 두고 UTF-8 사본 만들기

In [ ]:
with open(CSV, encoding="cp949") as src, open("nps_utf8.csv", "w", encoding="utf-8") as dst:
    for line in src:
        dst.write(line)

## Step 3. 모든 컬럼을 문자로 raw 테이블에 넣기

In [ ]:
import duckdb
con = duckdb.connect("nps.duckdb")
con.sql("""
CREATE OR REPLACE TABLE raw_nps AS
SELECT * FROM read_csv('nps_utf8.csv', header = true, all_varchar = true)
""")
print(con.sql("SELECT COUNT(*) AS rows, MIN(자료생성년월) AS ym_min, MAX(자료생성년월) AS ym_max FROM raw_nps"))

## Step 4. 의심할 다섯 가지

In [ ]:
print(con.sql("""
SELECT LENGTH(사업자등록번호) AS len, COUNT(*) AS rows,
       COUNT(DISTINCT 사업자등록번호) AS distinct_bizno
FROM raw_nps GROUP BY 1
"""))

In [ ]:
print(con.sql("""
SELECT 사업자등록번호, COUNT(*) AS 사업장수
FROM raw_nps GROUP BY 1 ORDER BY 2 DESC LIMIT 5
"""))

In [ ]:
print(con.sql("""
SELECT COUNT(*) AS 사업장수, SUM(CAST(가입자수 AS INT)) AS 가입자합계
FROM raw_nps
WHERE 사업자등록번호 = '837870' AND 사업장명 LIKE '쿠팡풀필먼트%'
"""))

In [ ]:
print(con.sql("""
SELECT "사업장가입상태코드 1 등록 2 탈퇴" AS 상태,
       COUNT(*) AS rows,
       SUM(CASE WHEN CAST(가입자수 AS INT) = 0 THEN 1 ELSE 0 END) AS 가입자0,
       SUM(CASE WHEN CAST(가입자수 AS INT) > 0 THEN 1 ELSE 0 END) AS 가입자있음
FROM raw_nps GROUP BY 1 ORDER BY 1
"""))

In [ ]:
print(con.sql("""
SELECT (SELECT COUNT(*) FROM raw_nps)
     - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM raw_nps)) AS 완전중복
"""))

## Step 5. 회사 하나 골라 보기

In [ ]:
print(con.sql("""
SELECT 사업장명, CAST(가입자수 AS INT) AS 가입자수,
       CAST(신규취득자수 AS INT) AS 신규, CAST(상실가입자수 AS INT) AS 상실
FROM raw_nps
WHERE 사업장명 IN ('주식회사 카카오', '쿠팡 주식회사', '주식회사 카카오뱅크')
"""))